# 07 — Correlation vs. Causation

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2 hours  
**Week 3 — Statistics for ML & Python Data Stack**

---

> *"Correlation does not imply causation" — the single most important sentence in data science.*

---

## Learning Objectives
By the end of this notebook you will be able to:
- Explain the difference between correlation and causation with concrete examples
- Compute and interpret Pearson and Spearman correlation coefficients
- Build and read a correlation matrix for feature selection
- Identify confounders, spurious correlations, and Simpson's Paradox
- Explain why Anscombe's Quartet should make you distrust a single number
- Avoid the #1 mistake data scientists make when interpreting model outputs

## 1. Why Does This Matter for Machine Learning?

Machine learning models are **correlation machines**. Every algorithm — linear regression, random forests, neural networks — works by finding statistical patterns (correlations) in data. This is their superpower.

But correlation is not causation. Confusing the two leads to:

| Mistake | Example | Consequence |
|---------|---------|-------------|
| Wrong intervention | "Model says ice cream sales predict drowning → ban ice cream" | Wasted resources, real problem unsolved |
| Wrong feature reasoning | "Chimneys correlate with price → build more chimneys" | Renovation budget wasted |
| Unfair models | "ZIP code predicts loan default" | May be a proxy for race (confounder) |
| Bad business decisions | "Hiring more engineers correlates with more bugs" | Engineers are hired BECAUSE of bugs, not causing them |

**The rule:** Use correlation to find *which features are useful*. Never use correlation alone to decide *what to change* in the real world.

## 2. The Big Analogy: Ice Cream and Drowning

Imagine you're a public health researcher. You collect monthly data for a city over 10 years:

```
Month        Ice Cream Sales ($)    Drowning Deaths
January      $12,000                2
February     $14,000                3
...          ...                    ...
July         $95,000                18
August       $88,000                16
...          ...                    ...
```

You compute the correlation: **r = 0.85**. Very strong!

**Wrong conclusion:** "Ice cream causes drowning. Ban ice cream to save lives."

**Right conclusion:** Both ice cream sales AND outdoor swimming (and thus drowning) are driven by a **hidden third variable**: **summer heat**.

```
        TEMPERATURE (Confounder)
           /                  \
          ↓                    ↓
  Ice Cream Sales ←?→ Drowning Deaths
  (NOT causally linked)   
```

This hidden variable is called a **confounder**. It causes both observed variables to move together, creating an *apparent* relationship that does not exist directly.

### The House Price Version

In housing data:
- **Shoe size** (of the owner) might correlate with **house price** in some dataset
- Why? Wealthy people can afford bigger houses AND more expensive shoes
- **Confounder = Income/Wealth**
- Conclusion: Don't recommend people buy bigger shoes to increase their house value!

## 3. What IS Correlation? Plain English First

**Correlation** measures how consistently two variables move together.

Think of it like dance partners:
- **Perfect positive correlation (r=1):** When one steps forward, the other ALWAYS steps forward by the same proportion. Every time, without fail.
- **Perfect negative correlation (r=-1):** When one steps forward, the other ALWAYS steps backward.
- **Zero correlation (r=0):** One partner's moves tell you nothing about the other's.
- **Partial correlation (0 < r < 1):** Usually they move together, but not always. Sometimes one zigs when the other zags.

### Important caveat: r=0 does NOT mean "no relationship"

r measures **linear** relationships only. Two variables can have a strong **curved** (nonlinear) relationship and still show r ≈ 0. We'll demonstrate this with house age below.

## 4. The Formula

The **Pearson correlation coefficient** between variables X and Y:

$$r = \frac{\text{cov}(X, Y)}{\sigma_X \cdot \sigma_Y}$$

Where:
- $\text{cov}(X, Y) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$ — covariance (do they vary together?)
- $\sigma_X$ = standard deviation of X (how spread out is X?)
- $\sigma_Y$ = standard deviation of Y

**Why divide by standard deviations?** To make the result unitless and always between -1 and +1. Without this normalization, two variables measured in different units (e.g., square feet vs. dollars) would give you incomparable numbers.

**Intuition:** Covariance tells you direction (positive or negative), but its magnitude depends on the scale of your variables. Dividing by the standard deviations standardizes it.

### Spearman Correlation

$$r_s = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$

Where $d_i$ is the difference in **ranks** of the two variables for each observation.

Spearman converts values to ranks first, then applies Pearson's formula. This makes it:
- Robust to outliers (one extreme value can't dominate)
- Sensitive to **monotonic** relationships (consistently increasing/decreasing, even if curved)
- Better for ordinal data (e.g., star ratings)

In [ ]:
# ============================================================
# SETUP: Import all libraries we'll need throughout
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set random seed for reproducibility — same synthetic data every run
np.random.seed(42)

# Set global plot style so all charts look clean and consistent
plt.rcParams['figure.figsize'] = (10, 6)  # default figure size
plt.rcParams['font.size'] = 12             # readable font size
sns.set_theme(style='whitegrid', palette='muted')  # seaborn theme

print("Libraries loaded successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# ============================================================
# STEP 1: Generate Synthetic House Price Dataset
# We'll build correlations in deliberately so we can study them
# ============================================================
n = 500  # number of houses in our dataset

# --- Base variable: house size (sq ft) ---
# Normal distribution: most houses are 1000–3000 sq ft
size = np.random.normal(loc=2000, scale=500, size=n)
size = np.clip(size, 600, 5000)  # no houses smaller than 600 or larger than 5000 sq ft

# --- Bedrooms: loosely tied to size ---
# More sq ft → tends to have more bedrooms, but with noise
bedrooms = np.round(size / 600 + np.random.normal(0, 0.5, n)).astype(int)
bedrooms = np.clip(bedrooms, 1, 8)  # keep bedrooms between 1 and 8

# --- Bathrooms: also tied to size ---
bathrooms = np.round(size / 900 + np.random.normal(0, 0.4, n)).astype(int)
bathrooms = np.clip(bathrooms, 1, 6)  # keep between 1 and 6

# --- House age (years): independent of size ---
age = np.random.uniform(0, 80, n)  # houses range from brand new to 80 years old

# --- Price: function of size, bathrooms, age, plus noise ---
# The TRUE data-generating process:
#   base price: $150/sqft
#   bathroom premium: $15,000 per bathroom
#   age penalty: -$1,500 per year (older = cheaper)
#   noise: random market variation ±$30,000
price = (
    150 * size                                    # size drives most of the price
    + 15000 * bathrooms                           # bathrooms add value
    - 1500 * age                                  # older houses are cheaper
    + np.random.normal(0, 30000, n)               # market noise
)
price = np.clip(price, 80000, 1200000)  # realistic price range

# --- Build a pandas DataFrame ---
df = pd.DataFrame({
    'size_sqft':  np.round(size).astype(int),
    'bedrooms':   bedrooms,
    'bathrooms':  bathrooms,
    'age_years':  np.round(age).astype(int),
    'price':      np.round(price).astype(int)
})

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# ============================================================
# STEP 2: Basic descriptive statistics
# Always examine your data before computing correlations
# ============================================================
print("=== Descriptive Statistics ===")
print(df.describe().round(2))

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())  # always check for nulls before computing correlations

In [ ]:
# ============================================================
# STEP 3: Compute Pearson Correlation Matrix
# pandas .corr() computes pairwise Pearson correlations
# ============================================================

# Compute the full correlation matrix
corr_matrix = df.corr(method='pearson')  # default is pearson

print("=== Pearson Correlation Matrix ===")
print(corr_matrix.round(3))

print("\n=== Correlations with PRICE specifically ===")
# Sort by absolute correlation with price to see which features matter most
price_corr = corr_matrix['price'].drop('price')  # drop self-correlation (always 1.0)
print(price_corr.sort_values(ascending=False).round(3))

print("\nInterpretation:")
print(f"  size_sqft vs price:  r = {corr_matrix.loc['size_sqft','price']:.3f}  → strong positive")
print(f"  bathrooms vs price:  r = {corr_matrix.loc['bathrooms','price']:.3f}  → moderate positive")
print(f"  bedrooms vs price:   r = {corr_matrix.loc['bedrooms','price']:.3f}  → moderate positive")
print(f"  age_years vs price:  r = {corr_matrix.loc['age_years','price']:.3f}  → weak negative")

In [ ]:
# ============================================================
# VISUALIZATION 1: Correlation Heatmap
# The standard way to visualize a correlation matrix
# ============================================================
fig, ax = plt.subplots(figsize=(8, 6))

# annot=True: show the correlation number in each cell
# fmt='.2f': format numbers to 2 decimal places
# cmap='coolwarm': blue=negative, red=positive, white=zero
# vmin/vmax: force color scale to -1 to +1
# linewidths: thin borders between cells for readability
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,  # make each cell square-shaped
    ax=ax
)

ax.set_title('House Price Dataset — Pearson Correlation Matrix\n'
             'Red = positive correlation | Blue = negative correlation | White = near zero',
             fontsize=11)
plt.tight_layout()
plt.show()

print("Reading the heatmap:")
print("  - Diagonal is always 1.0 (every variable is perfectly correlated with itself)")
print("  - Matrix is symmetric: corr(A,B) == corr(B,A)")
print("  - Look at the 'price' row/column to find useful features for ML")
print("  - Also look for correlated FEATURES (multicollinearity) — bedrooms & bathrooms are both red")

In [ ]:
# ============================================================
# VISUALIZATION 2: Scatter Plots for Each Feature vs Price
# See the actual pattern, not just the number
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Size vs Price (strong positive correlation) ---
r_size = df['size_sqft'].corr(df['price'])  # compute correlation
axes[0].scatter(df['size_sqft'], df['price'], alpha=0.3, color='steelblue', s=20)
# Add regression line to visualize the trend
m, b = np.polyfit(df['size_sqft'], df['price'], 1)  # fit a degree-1 polynomial (line)
x_range = np.linspace(df['size_sqft'].min(), df['size_sqft'].max(), 100)
axes[0].plot(x_range, m * x_range + b, color='red', linewidth=2, label='Trend line')
axes[0].set_xlabel('House Size (sq ft)')
axes[0].set_ylabel('Price ($)')
axes[0].set_title(f'Size vs Price\nr = {r_size:.3f} (strong positive)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# --- Plot 2: Bathrooms vs Price (moderate positive correlation) ---
r_bath = df['bathrooms'].corr(df['price'])
# Add jitter to x-axis so overlapping points are visible (bathrooms is discrete)
jitter = np.random.normal(0, 0.06, n)  # small random noise for bathrooms
axes[1].scatter(df['bathrooms'] + jitter, df['price'], alpha=0.3, color='coral', s=20)
axes[1].set_xlabel('Number of Bathrooms')
axes[1].set_ylabel('Price ($)')
axes[1].set_title(f'Bathrooms vs Price\nr = {r_bath:.3f} (moderate positive)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# --- Plot 3: Age vs Price (weak negative correlation) ---
r_age = df['age_years'].corr(df['price'])
axes[2].scatter(df['age_years'], df['price'], alpha=0.3, color='green', s=20)
m2, b2 = np.polyfit(df['age_years'], df['price'], 1)
x_range2 = np.linspace(df['age_years'].min(), df['age_years'].max(), 100)
axes[2].plot(x_range2, m2 * x_range2 + b2, color='red', linewidth=2)
axes[2].set_xlabel('House Age (years)')
axes[2].set_ylabel('Price ($)')
axes[2].set_title(f'Age vs Price\nr = {r_age:.3f} (weak negative)')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.suptitle('Feature Correlations with House Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice: The scatter shows that correlation is 'messier' in real data.")
print("The trend line captures the average direction, but individual points vary a lot.")

## 5. The Danger of r=0: Nonlinear Relationships

A Pearson correlation of r≈0 does NOT mean the variables are unrelated. It means there is no **linear** relationship.

**House Age Example — The Renovation Sweet Spot:**

Think about real housing markets:
- Very new houses (0–5 years): top price
- Mid-age houses (20–40 years): price dips — outdated finishes, some wear
- Old houses (50–80 years): price can be HIGH again if they become "vintage", "character homes"

This creates a **U-shaped (quadratic) relationship** between age and price. Pearson's r, which only measures linear trends, will be near zero — even though the relationship is strong!

**Lesson:** Always plot your data. Never trust a correlation number without seeing the scatter plot.

In [ ]:
# ============================================================
# DEMONSTRATION: r≈0 but strong NONLINEAR relationship
# House age with a U-shaped (quadratic) effect on price
# ============================================================

# Generate age with a U-shaped price effect
age_demo = np.linspace(0, 80, 500)  # ages 0–80 years

# U-shape: quadratic (x - 40)^2 creates a parabola with minimum at 40 years
# Coefficient tuned so the effect is realistic
price_nonlinear = (
    400000                              # baseline price
    + 3000 * (age_demo - 40)**2         # U-shape: dips at age=40, rises at extremes
    - 5000 * age_demo                   # slight overall downward trend
    + np.random.normal(0, 20000, 500)   # random noise
)
price_nonlinear = np.clip(price_nonlinear, 100000, 900000)

# Compute Pearson r — should be near 0 despite clear pattern
r_linear, p_linear = stats.pearsonr(age_demo, price_nonlinear)

# Compute Spearman r — rank-based, captures monotonic trends only
r_spearman, p_spearman = stats.spearmanr(age_demo, price_nonlinear)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: scatter with both linear and quadratic fit lines
axes[0].scatter(age_demo, price_nonlinear, alpha=0.3, color='purple', s=15, label='Data points')

# Linear fit (what Pearson r measures)
m_lin, b_lin = np.polyfit(age_demo, price_nonlinear, 1)  # degree-1 fit
x_plot = np.linspace(0, 80, 200)
axes[0].plot(x_plot, m_lin * x_plot + b_lin, 'r--', linewidth=2,
             label=f'Linear fit (r={r_linear:.3f})')

# Quadratic fit (what actually captures the pattern)
coeffs = np.polyfit(age_demo, price_nonlinear, 2)   # degree-2 polynomial
poly_fn = np.poly1d(coeffs)                         # create callable polynomial function
axes[0].plot(x_plot, poly_fn(x_plot), 'g-', linewidth=2.5, label='Quadratic fit (true shape)')

axes[0].set_xlabel('House Age (years)')
axes[0].set_ylabel('Price ($)')
axes[0].set_title(f'U-Shaped Relationship\nPearson r = {r_linear:.3f} ← misleadingly small!')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].legend()

# Right plot: residuals from linear fit — shows the missed pattern
residuals = price_nonlinear - (m_lin * age_demo + b_lin)
axes[1].scatter(age_demo, residuals, alpha=0.3, color='orange', s=15)
axes[1].axhline(0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('House Age (years)')
axes[1].set_ylabel('Residual ($ off from linear prediction)')
axes[1].set_title('Residuals from Linear Fit\nClear U-shape → linear model misses this')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.suptitle('r ≈ 0 Does NOT Mean "No Relationship"', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Pearson r  = {r_linear:.4f}  (p-value: {p_linear:.4f})")
print(f"Spearman r = {r_spearman:.4f}  (p-value: {p_spearman:.4f})")
print("\nBoth are near 0 — yet the quadratic pattern is clearly visible in the plot!")
print("ALWAYS visualize before trusting a correlation coefficient.")

## 6. Anscombe's Quartet — Same r, Completely Different Data

In 1973, statistician Francis Anscombe created four datasets that all have **identical** summary statistics:
- Same mean of X
- Same mean of Y
- Same variance of X and Y
- Same Pearson correlation (r ≈ 0.816)
- Same regression line

But when you plot them, they look **completely different**.

This was deliberately designed to show that **statistical summaries alone are insufficient** — you must always visualize your data.

We'll recreate this concept with house price data.

In [ ]:
# ============================================================
# ANSCOMBE'S QUARTET: Same correlation, wildly different data
# Using actual Anscombe dataset values
# ============================================================

# Anscombe's original four datasets — hand-crafted to have identical statistics
anscombe_data = {
    'I':   {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]},
    'II':  {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]},
    'III': {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]},
    'IV':  {'x': [8,8,8,8,8,8,8,19,8,8,8],
             'y': [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]}
}

# Compute statistics for each dataset — they should be nearly identical
print("=== Statistics for each dataset (they're almost all the same!) ===")
print(f"{'Dataset':>10} {'mean_x':>10} {'mean_y':>10} {'std_x':>10} {'std_y':>10} {'r':>10}")
print("-" * 60)

for name, data in anscombe_data.items():
    x, y = np.array(data['x']), np.array(data['y'])
    r_val, _ = stats.pearsonr(x, y)
    print(f"{'Dataset '+name:>10} {np.mean(x):>10.3f} {np.mean(y):>10.3f} "
          f"{np.std(x):>10.3f} {np.std(y):>10.3f} {r_val:>10.3f}")

# Plot all four datasets
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()  # flatten 2D array of axes into 1D for easy iteration

labels = ['I — Linear (clean)',
          'II — Curved (quadratic)',
          'III — Linear + outlier',
          'IV — Vertical cluster + outlier']

for idx, (name, data) in enumerate(anscombe_data.items()):
    x, y = np.array(data['x']), np.array(data['y'])
    r_val, _ = stats.pearsonr(x, y)

    # Scatter plot
    axes[idx].scatter(x, y, color='steelblue', s=80, zorder=3, edgecolors='navy', linewidth=0.5)

    # Regression line
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(min(x) - 0.5, max(x) + 0.5, 100)
    axes[idx].plot(x_line, m * x_line + b, color='red', linewidth=2, label=f'r = {r_val:.3f}')

    axes[idx].set_xlim(2, 20)
    axes[idx].set_ylim(2, 14)
    axes[idx].set_title(f'Dataset {name}: {labels[idx]}\n'
                        f'r = {r_val:.3f} | mean_x={np.mean(x):.1f}, mean_y={np.mean(y):.1f}',
                        fontsize=10)
    axes[idx].legend(fontsize=10)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle("Anscombe's Quartet: Same r ≈ 0.816, Same Regression Line, Completely Different Data!\n"
             "LESSON: Always visualize your data — correlation statistics can deceive.",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Spurious Correlations — When the Numbers Lie

Tyler Vigen's website (tylervigen.com) collects hilarious spurious correlations from real data:

| Variable A | Variable B | r |
|-----------|-----------|---|
| Per capita cheese consumption | Deaths by bedsheet tangling | 0.947 |
| Nicolas Cage movie releases | Pool drownings | 0.666 |
| Divorce rate in Maine | Margarine consumption per capita | 0.993 |
| US spending on science | Suicides by hanging | 0.992 |

Why do these exist? With enough variables and enough time periods, some pairs will **by chance** show high correlation — even with no real connection.

**The p-value problem (multiple comparisons):** If you test 1000 pairs of variables, you'd expect ~50 to show "significant" correlation at p<0.05 purely by chance (5% × 1000 = 50 false positives). This is called the **multiple comparisons problem** or **data dredging**.

**In ML:** If you have 100 features and just look for correlations with your target, some will appear significant by chance. This is why you need train/test splits and cross-validation.

In [ ]:
# ============================================================
# DEMONSTRATION: Spurious correlations from multiple testing
# Generate 50 random features — how many correlate with price by chance?
# ============================================================
np.random.seed(123)  # different seed to show a different run

n_obs = 100          # small dataset: more likely to find spurious correlations
n_random_features = 50  # number of completely random, meaningless features

# Generate random house prices (our target)
fake_price = np.random.normal(300000, 80000, n_obs)

# Generate 50 completely random features (no real relationship with price)
# Each feature is just white noise
random_features = np.random.normal(0, 1, (n_obs, n_random_features))

# Compute correlation of each random feature with price
correlations = []
p_values = []
for i in range(n_random_features):
    r_val, p_val = stats.pearsonr(random_features[:, i], fake_price)
    correlations.append(r_val)
    p_values.append(p_val)

correlations = np.array(correlations)
p_values = np.array(p_values)

# How many appear "significant" at p < 0.05?
significant = np.sum(p_values < 0.05)

print(f"Out of {n_random_features} COMPLETELY RANDOM features:")
print(f"  Appear 'significant' (p < 0.05): {significant} features")
print(f"  Expected by chance: ~{n_random_features * 0.05:.1f} features")
print(f"  Maximum |r| found: {np.max(np.abs(correlations)):.3f}")
print()
print("NONE of these correlations are real — they're all noise!")
print("This is why ML models need train/test splits and cross-validation.")

# Visualize the distribution of random correlations
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of correlation values
axes[0].hist(correlations, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0.2, color='red', linestyle='--', label='r = 0.2 threshold')
axes[0].axvline(-0.2, color='red', linestyle='--')
axes[0].set_xlabel('Pearson r value')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Correlations\nfor 50 RANDOM Features vs Price')
axes[0].legend()

# Scatter of p-values
colors = ['red' if p < 0.05 else 'steelblue' for p in p_values]
axes[1].scatter(range(n_random_features), p_values, c=colors, s=50, alpha=0.8)
axes[1].axhline(0.05, color='red', linestyle='--', linewidth=2, label='p = 0.05 threshold')
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('p-value')
axes[1].set_title(f'p-values for 50 Random Features\n'
                  f'Red dots = "significant" (n={significant}) — all FALSE positives!')
axes[1].legend()

plt.suptitle('Multiple Testing Problem: Some Correlations are Purely by Chance',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Simpson's Paradox — When Groups Reverse the Correlation

**Simpson's Paradox** occurs when a trend that appears in several groups **reverses** when the groups are combined. It's one of the most counterintuitive phenomena in statistics.

### House Price Example

Imagine we look at two neighborhoods:
- **Suburb:** Larger houses, lower prices per sq ft (land is cheap)
- **Downtown:** Smaller houses, much higher prices per sq ft (premium location)

**Within each neighborhood:** More square footage → higher price (positive correlation)

**Combined dataset:** The correlation could appear negative or weaker, because downtown houses are small AND expensive, while suburban houses are large AND relatively cheap.

In [ ]:
# ============================================================
# SIMPSON'S PARADOX with House Data
# Correlation reverses when neighborhoods are combined
# ============================================================
np.random.seed(7)

# --- Downtown neighborhood ---
# Small houses (800–1600 sqft), very high prices ($400K–$900K)
n_downtown = 150
downtown_size  = np.random.normal(1100, 200, n_downtown)
downtown_price = (
    600000                                        # high base price (location premium)
    + 50 * downtown_size                          # each sq ft adds some value
    + np.random.normal(0, 50000, n_downtown)      # noise
)

# --- Suburb neighborhood ---
# Large houses (2000–4000 sqft), moderate prices ($200K–$600K)
n_suburb = 150
suburb_size  = np.random.normal(2800, 400, n_suburb)
suburb_price = (
    100000                                        # lower base price
    + 80 * suburb_size                            # sq ft adds value
    + np.random.normal(0, 40000, n_suburb)        # noise
)

# --- Combine both neighborhoods ---
all_sizes  = np.concatenate([downtown_size, suburb_size])
all_prices = np.concatenate([downtown_price, suburb_price])
all_neighborhood = ['Downtown'] * n_downtown + ['Suburb'] * n_suburb

# Compute correlations
r_downtown, _ = stats.pearsonr(downtown_size, downtown_price)
r_suburb,   _ = stats.pearsonr(suburb_size, suburb_price)
r_combined, _ = stats.pearsonr(all_sizes, all_prices)

print("=== Simpson's Paradox: Correlation by Group ===")
print(f"  Downtown neighborhood only:  r = {r_downtown:.3f}  (positive — bigger → more expensive)")
print(f"  Suburb neighborhood only:    r = {r_suburb:.3f}  (positive — bigger → more expensive)")
print(f"  Combined (both together):    r = {r_combined:.3f}  (NEGATIVE or weaker — the paradox!)")
print()
print("  Both groups individually show: more size → higher price")
print("  Combined, the correlation appears different because location (the confounder)")
print("  overwhelms the size effect.")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Combined data (misleading view)
axes[0].scatter(all_sizes, all_prices, alpha=0.4, color='gray', s=20, label='All houses')
m_comb, b_comb = np.polyfit(all_sizes, all_prices, 1)
x_plot = np.linspace(all_sizes.min(), all_sizes.max(), 200)
axes[0].plot(x_plot, m_comb * x_plot + b_comb, 'r-', linewidth=2.5,
             label=f'Combined trend (r={r_combined:.2f})')
axes[0].set_xlabel('House Size (sq ft)')
axes[0].set_ylabel('Price ($)')
axes[0].set_title(f"Combined Data\nr = {r_combined:.3f}")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].legend(fontsize=9)

# Right: Colored by neighborhood (the true picture)
axes[1].scatter(downtown_size, downtown_price, alpha=0.5, color='steelblue', s=20, label='Downtown')
axes[1].scatter(suburb_size,   suburb_price,   alpha=0.5, color='coral',     s=20, label='Suburb')

# Trend lines for each group
m_d, b_d = np.polyfit(downtown_size, downtown_price, 1)
x_d = np.linspace(downtown_size.min(), downtown_size.max(), 100)
axes[1].plot(x_d, m_d * x_d + b_d, 'b-', linewidth=2.5,
             label=f'Downtown trend (r={r_downtown:.2f})')

m_s, b_s = np.polyfit(suburb_size, suburb_price, 1)
x_s = np.linspace(suburb_size.min(), suburb_size.max(), 100)
axes[1].plot(x_s, m_s * x_s + b_s, 'r-', linewidth=2.5,
             label=f'Suburb trend (r={r_suburb:.2f})')

axes[1].set_xlabel('House Size (sq ft)')
axes[1].set_ylabel('Price ($)')
axes[1].set_title(f"Colored by Neighborhood\nBoth show positive r — but combined showed {r_combined:.2f}!")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].legend(fontsize=9)

plt.suptitle("Simpson's Paradox: The group you ignore can flip your conclusion!",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Pearson vs. Spearman — When to Use Each

| Situation | Use Pearson | Use Spearman |
|-----------|------------|---------------|
| Both variables continuous, roughly normal | ✅ Yes | OK |
| One or both variables ordinal (e.g., star ratings) | ❌ No | ✅ Yes |
| Outliers present | ❌ Sensitive | ✅ Robust |
| Monotonic but curved relationship | ❌ May miss it | ✅ Captures it |
| Need to detect non-monotonic (U-shape) | ❌ No | ❌ No (use plots!) |
| Testing linear assumption | ✅ Directly | ❌ No |

**Practical advice:** Compute both, compare them. If Spearman >> Pearson, your data has outliers or a curved relationship. If they're similar, the relationship is roughly linear.

In [ ]:
# ============================================================
# Pearson vs Spearman: Effect of Outliers
# ============================================================
np.random.seed(42)

# Clean data: nice linear relationship between garage size and price
garage_sqft = np.random.normal(400, 80, 50)  # garage size in sq ft
price_clean = 200000 + 500 * garage_sqft + np.random.normal(0, 20000, 50)

# Add 3 extreme outliers: huge garages but very cheap houses (e.g., rural warehouses)
garage_outlier = np.array([1500, 1800, 2000])    # very large garages
price_outlier  = np.array([80000, 75000, 90000]) # but very cheap houses

# Combine
garage_all = np.concatenate([garage_sqft, garage_outlier])
price_all  = np.concatenate([price_clean, price_outlier])

# Compute correlations for clean vs. outlier-affected data
r_p_clean, _    = stats.pearsonr(garage_sqft, price_clean)
r_s_clean, _    = stats.spearmanr(garage_sqft, price_clean)
r_p_outliers, _ = stats.pearsonr(garage_all, price_all)
r_s_outliers, _ = stats.spearmanr(garage_all, price_all)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Clean data
axes[0].scatter(garage_sqft, price_clean, alpha=0.6, color='steelblue', s=40)
axes[0].set_xlabel('Garage Size (sq ft)')
axes[0].set_ylabel('Price ($)')
axes[0].set_title(f'Clean Data\nPearson r={r_p_clean:.3f} | Spearman r={r_s_clean:.3f}\n(Both agree)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Data with outliers
axes[1].scatter(garage_sqft, price_clean, alpha=0.6, color='steelblue', s=40, label='Normal houses')
axes[1].scatter(garage_outlier, price_outlier, alpha=0.9, color='red', s=120,
                marker='*', label='Outliers (rural warehouses)', zorder=5)
axes[1].set_xlabel('Garage Size (sq ft)')
axes[1].set_ylabel('Price ($)')
axes[1].set_title(f'With 3 Outliers Added\n'
                  f'Pearson r={r_p_outliers:.3f} | Spearman r={r_s_outliers:.3f}\n'
                  f'(Pearson drops more! Spearman more robust)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].legend(fontsize=9)

plt.suptitle('Pearson vs Spearman: Spearman is More Robust to Outliers',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Clean data:         Pearson={r_p_clean:.3f}, Spearman={r_s_clean:.3f}  → both agree")
print(f"With 3 outliers:    Pearson={r_p_outliers:.3f}, Spearman={r_s_outliers:.3f}  → Pearson hurt more")
print("\nSpearman is based on RANKS, so a single extreme value can only shift one rank,")
print("not drag the entire statistic.")

## 10. Using Correlation for Feature Selection in ML

In practice, you use correlation to:

1. **Find useful features:** High |r| with target → good candidate for a model
2. **Detect multicollinearity:** Features correlated with each other cause problems in linear models (the model can't distinguish which feature matters)
3. **Reduce redundancy:** If two features have r=0.95, you probably only need one

**Practical rule of thumb:**
- |r| < 0.1 → very weak, probably useless (but check for nonlinearity!)
- 0.1 ≤ |r| < 0.3 → weak but potentially useful
- 0.3 ≤ |r| < 0.5 → moderate
- 0.5 ≤ |r| < 0.7 → strong
- |r| ≥ 0.7 → very strong (also check if they're basically the same feature)

In [ ]:
# ============================================================
# FEATURE SELECTION using Correlation
# Identify useful features and multicollinear pairs
# ============================================================

# Add a few more features to our house dataset to make this interesting
np.random.seed(42)

# Total rooms (= bedrooms + bathrooms + 2 other rooms, with noise) — highly correlated with bedrooms
df['total_rooms'] = df['bedrooms'] + df['bathrooms'] + np.random.randint(2, 5, n)

# Lot size: weakly related to house size
df['lot_size_sqft'] = df['size_sqft'] * np.random.uniform(2, 8, n) + np.random.normal(0, 1000, n)
df['lot_size_sqft'] = df['lot_size_sqft'].clip(1000, 40000).astype(int)

# Distance from city center (km): inversely related to price
df['dist_city_km'] = np.random.exponential(scale=12, size=n)
df['price'] = df['price'] - 3000 * df['dist_city_km'] + np.random.normal(0, 10000, n)
df['price'] = df['price'].clip(50000, 1500000).astype(int)

# Recompute correlation matrix with all features
corr_extended = df.corr(method='pearson')

# --- Plot 1: Full heatmap ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(corr_extended, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=axes[0])
axes[0].set_title('Extended Correlation Matrix (all features)', fontsize=11)

# --- Plot 2: Bar chart of correlations with price ---
price_corr2 = corr_extended['price'].drop('price').sort_values()
colors2 = ['coral' if v < 0 else 'steelblue' for v in price_corr2]
axes[1].barh(price_corr2.index, price_corr2.values, color=colors2, edgecolor='gray')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].axvline(0.3,  color='green', linestyle='--', alpha=0.7, label='|r|=0.3 threshold')
axes[1].axvline(-0.3, color='green', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Pearson r with Price')
axes[1].set_title('Feature Correlations with Price\n(for feature selection)')
axes[1].legend()

# Add value labels on the bars
for i, (val, name) in enumerate(zip(price_corr2.values, price_corr2.index)):
    axes[1].text(val + 0.01 * np.sign(val), i, f'{val:.2f}',
                 va='center', fontsize=10)

plt.tight_layout()
plt.show()

# Identify highly correlated FEATURE PAIRS (multicollinearity)
print("=== Highly Correlated Feature Pairs (|r| > 0.6, excluding price) ===")
feature_cols = [c for c in df.columns if c != 'price']
corr_features = df[feature_cols].corr()

for i in range(len(feature_cols)):
    for j in range(i+1, len(feature_cols)):  # upper triangle only (avoid duplicates)
        r_val = corr_features.iloc[i, j]
        if abs(r_val) > 0.6:
            print(f"  {feature_cols[i]:20s} ↔ {feature_cols[j]:20s}  r = {r_val:.3f}")
            print(f"    → These features are redundant. Consider dropping one.")

## 11. Self-Check Questions

Test your understanding before moving on.

---

### Question 1
The correlation between square footage and house price is r = 0.85. A real estate developer says: "This proves that making a house bigger CAUSES the price to rise. We should always add square footage to increase value." 

Is this reasoning correct? What else might explain the correlation?

<details>
<summary>Click for Answer</summary>

**No, this is not correct reasoning.** r=0.85 tells us these variables move together strongly, but it does not establish causation.

**Alternative explanations:**
- **Reverse causation:** Owners of high-value properties *choose* to build larger (expensive land → build big to justify it)
- **Confounders:** Wealthy neighborhoods have both larger AND more expensive homes — neighborhood quality causes both
- **Omitted variable bias:** Renovation quality, kitchen upgrades, school district quality all affect price AND tend to co-occur with size
- **Context matters:** Adding 200 sq ft of unfinished basement may not raise price at all

**To establish causation, you'd need:** a randomized experiment (randomly assign additions to houses) or a natural experiment (e.g., a zoning change that forced some houses to expand).
</details>

---

### Question 2
You compute r = 0.02 between house age and price. A colleague says: "Great, house age is useless — no relationship." Is your colleague right? What might they be missing?

<details>
<summary>Click for Answer</summary>

**Your colleague is wrong.** r ≈ 0 only means there is no *linear* relationship.

**What they might be missing:**
- A **U-shaped (quadratic) relationship**: brand new houses and very old "vintage" homes are both expensive; mid-age houses are cheaper. This would give r ≈ 0 even though age strongly predicts price.
- A **threshold effect**: houses under 5 years old have a massive premium, but age doesn't matter much beyond that.
- **Simpson's Paradox**: in different neighborhoods, age might have opposite effects.

**Always plot age vs. price as a scatter before concluding there's no relationship.**
</details>

---

### Question 3
Your ML model finds that "number of chimneys" is strongly predictive of house price (r = 0.72). Your manager concludes: "Chimneys cause higher prices — let's add chimneys to every house we flip." What's more likely happening? What would you tell your manager?

<details>
<summary>Click for Answer</summary>

**This is a classic confounder situation.** More likely:

- Chimneys are common in **large, older, upscale homes** — which are expensive for entirely different reasons (size, quality, location, prestige)
- The **confounder** is house quality/size/age — it causes both more chimneys AND higher prices
- Adding a chimney to a cheap house will NOT transform it into an expensive one

**What to tell your manager:** "The model found a correlation, not a cause. Chimneys are a *signal* of expensive homes, not a *driver* of price. Adding chimneys to cheap homes won't replicate the full profile of luxury properties that naturally have chimneys. We'd likely spend $15,000 on a chimney that adds only marginal value."

**Lesson:** ML features tell you what to *look for*, not what to *change*.
</details>

---

### Question 4
You're analyzing house prices and find that the correlation between `total_rooms` and `bedrooms` is r = 0.93. Both are correlated with price (r ≈ 0.5). Should you include both in your linear regression model? Why or why not?

<details>
<summary>Click for Answer</summary>

**No — you should avoid including both.** This is called **multicollinearity**.

**Problems it causes in linear regression:**
- The model can't separate the effect of `bedrooms` from `total_rooms` (they move almost identically)
- Coefficient estimates become unstable — small changes in data lead to wildly different coefficients
- Standard errors inflate → p-values become unreliable
- The model may assign bizarre coefficients (e.g., bedrooms: +$50,000 and total_rooms: -$45,000)

**Solution:** Keep just one (usually the more interpretable one), or create a single combined feature. Note: tree-based models (random forests, XGBoost) are less affected by multicollinearity.
</details>

---

### Question 5
Describe Anscombe's Quartet in one sentence. Why was it created, and what should it change about your data analysis workflow?

<details>
<summary>Click for Answer</summary>

**One sentence:** Anscombe's Quartet is four datasets with identical means, variances, and correlations that look completely different when plotted — one linear, one curved, one with an outlier, one with a vertical cluster.

**Why created:** Francis Anscombe (1973) designed it to demonstrate that statistical summaries alone are insufficient for data analysis, specifically targeting the then-growing overreliance on computational statistics without visualization.

**What it should change:** Always plot your data before trusting any statistical summary. A correlation matrix tells you correlation *values*, not the *shape* of relationships. Add scatter plots and residual plots to your standard exploratory data analysis (EDA) workflow.
</details>

## 12. Key Takeaways

| Concept | What it means | Common mistake |
|---------|--------------|----------------|
| **Correlation** | Two variables tend to move together | Thinking it implies causation |
| **Causation** | One variable makes another change | Concluding from observational data |
| **Confounder** | Hidden variable causing apparent correlation | Ignoring it → wrong interventions |
| **Pearson r** | Measures linear relationships, sensitive to outliers | Trusting it without plotting |
| **Spearman r** | Rank-based, handles curved/outlier data | Using Pearson when data has outliers |
| **r=0** | No linear relationship | Concluding "no relationship at all" |
| **Anscombe's Quartet** | Same r can mean very different patterns | Trusting summary stats without plotting |
| **Simpson's Paradox** | Group-level trends can reverse in aggregate | Ignoring subgroup analysis |
| **Spurious correlation** | Random chance in large variable sets | Not using train/test splits |
| **Multicollinearity** | Correlated features cause linear model instability | Including all correlated features |

---

**Golden rule for ML practitioners:** Correlation tells you which features to include in your model. It does not tell you which variables to intervene on in the real world. For that, you need domain knowledge, controlled experiments, or causal inference methods (Pearl's do-calculus, difference-in-differences, instrumental variables).

---

*Next notebook: 08 — NumPy: the engine under every ML framework*